# Gold `fact_province_year`

One row per reporting unit and year (100 × 6 = 600). It brings tourism, economy and population together so the views can compare them.

| | |
| --- | --- |
| **Key** | (`province_psgc`, `year`) |
| **Foreign keys** | `dim_province.province_psgc`, `dim_year.year` |
| **Built from** | `dot_travelers_unit_year`, `psa_economy_unit_year`, `population_unit_year` |
| **Requires** | `dim_province` and `dim_year` already built |

**Rules**

* The grain is every unit × every year. A year a source does not cover is NULL, never 0 (DL-009).
* Columns are copied from Silver; nothing is recomputed. Ratios, shares and growth belong to the views.
* `comparable_to_2019` is year-specific and comes from DOT (DL-013, DL-020).
* `traveler_value_status` is never NULL for a DOT year. When it is `not_reported`, all four traveler columns are NULL.
* Units are written as column comments from `ref_indicator` (D17).

**Run log:** the run is `STARTED` first and is marked `SUCCESS` or `SUCCESS_WITH_REVIEW` only after the build, checks, write **and** column comments all finish. Any error marks this `run_id` as `FAILED`, records which tables were already written, saves the error text, and re-raises the original error.

## Configuration, helpers and run log

`dq(...)` records each check. `update_run_log(...)` and `save_dq_results()` are separate helpers so the error handler in Step 4 can use them. The row and table counts in the run log come from what was actually written.

In [0]:
from pyspark.sql import functions as F
from functools import reduce
from types import SimpleNamespace
import uuid
from datetime import datetime

CATALOG = "tuklas_dev"
SILVER_SCHEMA = "03_silver"
GOLD_SCHEMA = "04_gold"
DQ_SCHEMA = "06_data_quality"
NOTEBOOK_NAME = "03_gold_fact_province_year"
run_id = f"gold_fact_province_year_{datetime.now().strftime('%Y%m%d_%H%M%S')}_{uuid.uuid4().hex[:8]}"

TBL_FACT = "fact_province_year"
EXPECTED_UNITS, YEARS, BASELINE_YEAR = 100, [2019, 2020, 2021, 2022, 2023, 2024], 2019
VALID_STATUS = {"reported", "dash_as_zero", "reported_partial", "not_reported"}   # DL-019
EXPECTED_UNITS_WITHOUT_DOT = {"0906600000", "1900700000", "1903600000", "1907000000", "1908700000", "1908800000"}
TOUR_COLS = ["foreign_travelers", "overseas_filipinos", "domestic_travelers", "total_travelers"]
ECON_COLS = ["gdp_total_current", "gdp_total_constant", "gdp_per_capita_current", "afs_gva_current", "afs_gva_constant"]

def silver_table(n): return f"`{CATALOG}`.`{SILVER_SCHEMA}`.`{n}`"
def gold_table(n):   return f"`{CATALOG}`.`{GOLD_SCHEMA}`.`{n}`"
def dq_table(n):     return f"`{CATALOG}`.`{DQ_SCHEMA}`.`{n}`"

# ── Run state ────────────────────────────────────────────────────
dq_results = []
written = {}          # Gold tables fully written in this run: name -> rows
dq_saved = False

def dq(check_id, table_name, severity, failed_rows, expected, observed, sample_keys=None, notes=""):
    """Record one data-quality check result. failed_rows = 0 means PASS."""
    status = "PASS" if failed_rows == 0 else "FAIL"
    keys = " | ".join(str(k) for k in list(sample_keys)[:20]) if sample_keys else None
    dq_results.append((run_id, NOTEBOOK_NAME, check_id, table_name, severity, status, int(failed_rows),
                       str(expected), str(observed), keys, notes))
    marker = "✓" if status == "PASS" else ("✗" if severity == "BLOCK" else "!")
    print(f"{marker} {check_id:<11} {severity:<6} {status:<4} | {table_name} | expected {expected} | observed {observed}"
          + (f" | e.g. {keys}" if keys else ""))

def sample(df, *cols, n=20):
    return [" / ".join("" if r[c] is None else str(r[c]) for c in cols) for r in df.select(*cols).limit(n).collect()]

def dq_counts():
    blocks = [r for r in dq_results if r[4] == "BLOCK" and r[5] == "FAIL"]
    reviews = [r for r in dq_results if r[4] == "REVIEW" and r[5] == "FAIL"]
    return blocks, reviews

def update_run_log(status, notes):
    """Update this run_id's row. Table and row counts come from what was actually written so far."""
    blocks, reviews = dq_counts()
    spark.sql(f"""
    UPDATE {gold_table('gold_run_log')}
    SET status = :status, completed_at = current_timestamp(), tables_written = :tw, rows_written = :rw,
        dq_checks_run = :dc, dq_block_failures = :db, dq_review_flags = :dr, notes = :notes
    WHERE run_id = :run_id
    """, args={"status": status, "tw": len(written), "rw": sum(written.values()), "dc": len(dq_results),
               "db": len(blocks), "dr": len(reviews), "notes": notes[:2000], "run_id": run_id})

def save_dq_results():
    """Append this run's check results to dq_results once, even when the run stops early."""
    global dq_saved
    if dq_saved or not dq_results:
        return
    dq_df = (spark.createDataFrame(dq_results,
        "run_id STRING, notebook_name STRING, check_id STRING, table_name STRING, severity STRING, status STRING, "
        "failed_rows BIGINT, expected STRING, observed STRING, sample_keys STRING, notes STRING")
        .withColumn("checked_at", F.current_timestamp()))
    display(dq_df.select("check_id", "severity", "status", "failed_rows", "expected", "observed", "sample_keys"))
    dq_df.select("run_id", "notebook_name", "check_id", "table_name", "severity", "status", "failed_rows",
                 "expected", "observed", "sample_keys", "checked_at", "notes") \
         .write.format("delta").mode("append").saveAsTable(dq_table("dq_results"))
    dq_saved = True

spark.sql(f"""
INSERT INTO {gold_table('gold_run_log')}
(run_id, notebook_name, started_at, completed_at, tables_written, rows_written,
 dq_checks_run, dq_block_failures, dq_review_flags, status, notes)
VALUES (:run_id, :nb, current_timestamp(), NULL, 0, 0, 0, 0, 0, 'STARTED', 'Notebook execution started')
""", args={"run_id": run_id, "nb": NOTEBOOK_NAME})
print(f"Run ID: {run_id}")

## Step 1: Build

The grid is `dim_province` × `dim_year`. Each Silver table is joined on (`unit_psgc_code`, `year`). Silver already has one row per unit and year, so the joins cannot add rows.

In [0]:
def load_inputs():
    return SimpleNamespace(
        dim_province=spark.table(gold_table("dim_province")),
        dim_year=spark.table(gold_table("dim_year")),
        dot=spark.table(silver_table("dot_travelers_unit_year")),
        econ=spark.table(silver_table("psa_economy_unit_year")),
        pop=spark.table(silver_table("population_unit_year")),
        ref=spark.table(silver_table("ref_indicator")))

def build_fact(src):
    grid = src.dim_province.select(F.col("province_psgc").alias("unit_psgc_code")).crossJoin(src.dim_year.select("year"))
    d = src.dot.select("unit_psgc_code", "year", *TOUR_COLS, "traveler_value_status", "comparable_to_2019")
    e = src.econ.select("unit_psgc_code", "year", *ECON_COLS)
    p = src.pop.select("unit_psgc_code", "year", "population_total", "population_source")
    fact = (grid.join(d, ["unit_psgc_code", "year"], "left")
                .join(e, ["unit_psgc_code", "year"], "left")
                .join(p, ["unit_psgc_code", "year"], "left")
                .withColumnRenamed("unit_psgc_code", "province_psgc")
                .withColumn("built_at", F.current_timestamp())
                .select("province_psgc", "year", *TOUR_COLS, "traveler_value_status", *ECON_COLS,
                        "population_total", "population_source", "comparable_to_2019", "built_at"))
    display(fact.groupBy("year").agg(F.count("*").alias("rows"), F.count("total_travelers").alias("with_travelers"),
            F.count("gdp_total_current").alias("with_gdp"), F.count("population_total").alias("with_population")).orderBy("year"))
    return fact

## Step 2 — Data-quality checks

| ID | Check | Severity |
| --- | --- | --- |
| DQ-GLD-03 | 600 rows, one per (province, year) | BLOCK |
| DQ-GLD-04 | Every key exists in the dimensions | BLOCK |
| DQ-GLD-05 | Tourism columns, `traveler_value_status` and `comparable_to_2019` equal Silver DOT, row by row | BLOCK |
| DQ-GLD-06 | Economy columns equal Silver PSA, row by row | BLOCK |
| DQ-GLD-07 | `population_total` and `population_source` equal Silver population, row by row | BLOCK |
| DQ-GLD-08 | `traveler_value_status` uses the DL-019 values, is never NULL in a DOT year, and `not_reported` rows have all four traveler columns NULL | BLOCK |
| DQ-GLD-09 | `total_travelers` ≥ foreign, overseas Filipinos and domestic | REVIEW |
| DQ-GLD-10 | A&F GVA ≤ GDP (current and constant) | BLOCK |
| DQ-GLD-11 | Units without DOT rows have status `not_reported` (never NULL) and all four traveler columns NULL | BLOCK |
| DQ-GLD-12 | A source's columns are NULL outside its years | BLOCK |
| DQ-GLD-13 | `comparable_to_2019` is true (never NULL) in 2019 for every unit DOT reports | BLOCK |
| DQ-GLD-14 | Every `ref_indicator` measure is a fact column | BLOCK |

Fact-to-Silver comparisons (05–07) use `eqNullSafe`: NULL equals NULL, NULL never equals a value, and no NULL is turned into 0.

In [0]:
def compare_to_silver(fact, check_id, silver_df, cols, source_label):
    """Row-by-row on (province_psgc, year). eqNullSafe: NULL = NULL passes, NULL vs a value fails; no NULL becomes 0."""
    s = silver_df.select(F.col("unit_psgc_code").alias("province_psgc"), "year",
                         *[F.col(c).alias(f"_s_{c}") for c in cols])
    joined = fact.select("province_psgc", "year", *cols).join(s, ["province_psgc", "year"], "inner")
    missing = (s.select("province_psgc", "year")
                .join(fact.select("province_psgc", "year"), ["province_psgc", "year"], "left_anti"))
    differs = reduce(lambda a, b: a | b, [~F.col(c).eqNullSafe(F.col(f"_s_{c}")) for c in cols])
    bad = joined.filter(differs)
    bad_cols = [c for c in cols if bad.filter(~F.col(c).eqNullSafe(F.col(f"_s_{c}"))).limit(1).count() > 0]
    n_silver, n_joined = silver_df.count(), joined.count()
    failed = bad.count() + missing.count() + (0 if n_silver == n_joined else 1)
    dq(check_id, TBL_FACT, "BLOCK", failed,
       f"every {source_label} row matches the fact on (province, year), column by column",
       f"{bad.count()} rows differ ({', '.join(bad_cols) or 'none'}), {missing.count()} Silver rows missing, "
       f"{n_joined}/{n_silver} rows matched",
       sample(bad, "province_psgc", "year") + sample(missing, "province_psgc", "year"))

def run_checks(fact, src):
    yr = lambda df: sorted({r[0] for r in df.select("year").distinct().collect()})
    dot_years, econ_years, pop_years = yr(src.dot), yr(src.econ), yr(src.pop)
    all_null = reduce(lambda a, b: a & b, [F.col(c).isNull() for c in TOUR_COLS])

    # DQ-GLD-03  grain
    n = fact.count()
    dup = fact.groupBy("province_psgc", "year").count().filter("count > 1")
    dq("DQ-GLD-03", TBL_FACT, "BLOCK", dup.count() + (0 if n == EXPECTED_UNITS * len(YEARS) else 1),
       f"{EXPECTED_UNITS} x {len(YEARS)} = {EXPECTED_UNITS * len(YEARS)} rows, 0 duplicates",
       f"{n} rows, {dup.count()} duplicates", sample(dup, "province_psgc", "year"))

    # DQ-GLD-04  keys exist in the dimensions
    orph_p = fact.join(src.dim_province.select("province_psgc"), "province_psgc", "left_anti")
    orph_y = fact.join(src.dim_year.select("year"), "year", "left_anti")
    dq("DQ-GLD-04", TBL_FACT, "BLOCK", orph_p.count() + orph_y.count(), "every key in the dimensions",
       f"{orph_p.count()} unknown provinces, {orph_y.count()} unknown years")

    # DQ-GLD-05..07  fact equals Silver, row by row
    compare_to_silver(fact, "DQ-GLD-05", src.dot, TOUR_COLS + ["traveler_value_status", "comparable_to_2019"], "Silver DOT")
    compare_to_silver(fact, "DQ-GLD-06", src.econ, ECON_COLS, "Silver PSA economy")
    compare_to_silver(fact, "DQ-GLD-07", src.pop, ["population_total", "population_source"], "Silver population")

    # DQ-GLD-08  status: valid values, never NULL in a DOT year, not_reported means all four columns NULL
    bad_status = fact.filter(F.col("traveler_value_status").isNotNull() & ~F.col("traveler_value_status").isin(*VALID_STATUS))
    null_status = fact.filter(F.col("year").isin(*dot_years) & F.col("traveler_value_status").isNull())
    nr_with_value = fact.filter(F.col("traveler_value_status").eqNullSafe(F.lit("not_reported")) & ~all_null)
    dq("DQ-GLD-08", TBL_FACT, "BLOCK", bad_status.count() + null_status.count() + nr_with_value.count(),
       "DL-019 statuses; no NULL status in DOT years; not_reported rows have all four traveler columns NULL",
       f"{bad_status.count()} bad statuses, {null_status.count()} NULL statuses, "
       f"{nr_with_value.count()} not_reported rows with a value",
       sample(bad_status.unionByName(null_status).unionByName(nr_with_value), "province_psgc", "year", "traveler_value_status"))

    # DQ-GLD-09  total vs each component
    components = ["foreign_travelers", "overseas_filipinos", "domestic_travelers"]
    parts = fact.filter(reduce(lambda a, b: a | b, [F.col("total_travelers") < F.col(c) for c in components]))
    dq("DQ-GLD-09", TBL_FACT, "REVIEW", parts.count(), "total >= foreign, overseas Filipinos and domestic",
       f"{parts.count()} violations", sample(parts, "province_psgc", "year"))

    # DQ-GLD-10  A&F GVA <= GDP
    over = fact.filter((F.col("afs_gva_current") > F.col("gdp_total_current")) | (F.col("afs_gva_constant") > F.col("gdp_total_constant")))
    dq("DQ-GLD-10", TBL_FACT, "BLOCK", over.count(), "A&F GVA <= GDP", f"{over.count()} violations", sample(over, "province_psgc", "year"))

    # DQ-GLD-11  units DOT does not report: explicit not_reported (never NULL), all four traveler columns NULL
    no_dot = fact.filter(F.col("province_psgc").isin(*EXPECTED_UNITS_WITHOUT_DOT))
    bad_missing = no_dot.filter(~F.col("traveler_value_status").eqNullSafe(F.lit("not_reported")) | ~all_null)
    n_null_status = no_dot.filter(F.col("traveler_value_status").isNull()).count()
    dq("DQ-GLD-11", TBL_FACT, "BLOCK", bad_missing.count(),
       "units without DOT rows have status not_reported (never NULL) and all four traveler columns NULL",
       f"{bad_missing.count()} rows fail ({n_null_status} with a NULL status)",
       sample(bad_missing, "province_psgc", "year", "traveler_value_status"))

    # DQ-GLD-12  a source's columns are NULL outside its years
    def outside(cols, years):
        return fact.filter(~F.col("year").isin(*years) & reduce(lambda a, b: a | b, [F.col(c).isNotNull() for c in cols])).count()
    n_out = outside(ECON_COLS, econ_years) + outside(["population_total"], pop_years) + outside(TOUR_COLS, dot_years)
    dq("DQ-GLD-12", TBL_FACT, "BLOCK", n_out, "a source's columns are NULL outside its years", f"{n_out} values outside range")

    # DQ-GLD-13  every DOT unit is comparable with itself in the baseline year (NULL counts as a failure)
    bad_base = fact.filter((F.col("year") == BASELINE_YEAR)
                           & ~F.col("province_psgc").isin(*EXPECTED_UNITS_WITHOUT_DOT)
                           & ~F.coalesce(F.col("comparable_to_2019"), F.lit(False)))
    dq("DQ-GLD-13", TBL_FACT, "BLOCK", bad_base.count(),
       f"comparable_to_2019 = true in {BASELINE_YEAR} for every unit DOT reports",
       f"{bad_base.count()} units", sample(bad_base, "province_psgc", "comparable_to_2019"))

    # DQ-GLD-14  every ref_indicator measure is a fact column
    missing_cols = [r[0] for r in src.ref.select("measure").collect() if r[0] not in fact.columns]
    dq("DQ-GLD-14", TBL_FACT, "BLOCK", len(missing_cols), "every ref_indicator measure is a fact column",
       f"missing: {missing_cols}" if missing_cols else "all present", missing_cols)

## Step 3: Write and add column comments

Units come from `ref_indicator`, so they cannot drift from Silver. The run is not marked successful until the comments are done.

In [0]:
def write_fact(fact):
    (fact.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(gold_table(TBL_FACT)))
    written[TBL_FACT] = spark.table(gold_table(TBL_FACT)).count()
    print(f"✓ {gold_table(TBL_FACT)}: {written[TBL_FACT]} rows")

def add_column_comments(src):
    FACT = gold_table(TBL_FACT)
    for r in src.ref.collect():
        note = f"{r['title']} | unit: {r['unit']} | {r['price_basis']} | {r['additivity']}".replace("'", "''")
        spark.sql(f"ALTER TABLE {FACT} ALTER COLUMN `{r['measure']}` COMMENT '{note}'")
    for c in TOUR_COLS:
        spark.sql(f"ALTER TABLE {FACT} ALTER COLUMN `{c}` COMMENT 'DOT traveler count, as reported (persons)'")
    spark.sql(f"ALTER TABLE {FACT} ALTER COLUMN `population_total` COMMENT 'Population, persons; source in population_source'")
    spark.sql(f"ALTER TABLE {FACT} ALTER COLUMN `traveler_value_status` COMMENT 'reported | dash_as_zero | reported_partial | not_reported (DL-019). not_reported means all four traveler columns are NULL'")
    spark.sql(f"ALTER TABLE {FACT} ALTER COLUMN `comparable_to_2019` COMMENT 'true if DOT counts this unit the same way as in 2019 (DL-013, DL-020)'")
    print("✓ Column comments added.")

## Step 4: Run

STARTED → build → checks → write → column comments → SUCCESS. Any error marks this run `FAILED`, records which tables were already written, saves the error text, and re-raises the original error.

In [0]:
try:
    src = load_inputs()
    fact = build_fact(src)
    run_checks(fact, src)
    save_dq_results()

    blocks, _ = dq_counts()
    if blocks:
        raise AssertionError("BLOCK checks failed: " + ", ".join(r[2] for r in blocks)
                             + f". See {dq_table('dq_results')} for run_id = '{run_id}'.")

    write_fact(fact)
    add_column_comments(src)

    _, reviews = dq_counts()
    final_status = "SUCCESS_WITH_REVIEW" if reviews else "SUCCESS"
    notes = (f"{', '.join(f'{k}={v}' for k, v in written.items())}; {len(dq_results)} checks, {len(reviews)} for review"
             + (f" ({', '.join(r[2] for r in reviews)})" if reviews else "") + ".")
    update_run_log(final_status, notes)
    print(f"Run {run_id}: {final_status}\n{notes}")

except Exception as e:
    after = (f"Tables already written: {', '.join(f'{k}={v} rows' for k, v in written.items())}."
             if written else "No Gold tables written.")
    try:
        save_dq_results()                                   # keep any checks that already ran
        update_run_log("FAILED", f"{type(e).__name__}: {e} {after}")
    except Exception as log_err:                            # never hide the original error
        print(f"Could not update the run log: {log_err}")
    raise